In [5]:
# Importing the Libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler,LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor

# load the dataset
data = pd.read_excel('BBDM Project for 2nd research article.xlsx')

# Columns to exclude
columns_to_exclude = ['Chainage','Formation','RMC', ]

# Preprocessing: Exclude specified columns
X = data.drop(['PRnet'] + columns_to_exclude, axis=1)
Y = data['PRnet']

# Label encoding for multiple columns
label_encoder = LabelEncoder()
for col in ['Lithology', 'Weathering', 'Rock Strength']:
    X[col] = label_encoder.fit_transform(X[col])

# Train Test Split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)  # Removed 'stratify' as it is not used in regression

# Data Standardization
scaler = RobustScaler()
scaler.fit(X_train)  # Fit on the training data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define the Decision Tree Regressor
regressor = DecisionTreeRegressor(random_state=42)

# Define the hyperparameters you want to tune
param_grid = {
    'criterion': ['squared_error', 'friedman_mse', 'absolute_error'],
    'splitter': ['best', 'random'],
    'max_depth': [None, 2, 5, 10, 15, 20],
    'min_samples_split': list(range(1, 11)),
    'min_samples_leaf': list(range(1, 11)),
    'max_features': [None, 'sqrt', 'log2'],
}

# Create a grid search cross-validation object
grid_search = GridSearchCV(regressor, param_grid, cv=5, 
                           scoring='neg_mean_absolute_error',
                           n_jobs=-1, verbose=2)

# Fit the grid search to the data
grid_search.fit(X_train_scaled, Y_train)

# Print the best hyperparameters
print("Best Hyperparameters:", grid_search.best_params_)
# Evaluate the best model
best_regressor = grid_search.best_estimator_
Y_pred = best_regressor.predict(X_test_scaled)

# Calculate evaluation metrics
mae = mean_absolute_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

print("Mean Absolute Error:", mae)
print("R^2 Score:", r2)

Fitting 10 folds for each of 2688 candidates, totalling 26880 fits


C:\Users\tekbk\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_validation.py:425: FitFailedWarning: 
6720 fits failed out of a total of 26880.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3341 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\tekbk\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_validation.py", line 729, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\tekbk\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py", line 1145, in wrapper
    estimator._validate_params()
  File "C:\Users\tekbk\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py", line 638, in _validate_params
    

Best Hyperparameters: {'criterion': 'absolute_error', 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 6, 'min_samples_split': 2, 'splitter': 'best'}
Mean Absolute Error: 2.4352872896111433
R^2 Score: 0.925825572377502
